<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Training').getOrCreate()
sc = spark.sparkContext

In [2]:
emp_df = spark.read.format('csv').options(delimiter=',', header=True).load('/content/emp.csv')

In [3]:
dept_df = spark.read.format('csv').options(delimiter=',', header=True).load('/content/dept.csv')

In [4]:
emp_df.show()


+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|          1|          101|     John Doe| 30|  Male| 50000|01-01-2015|
|          2|          101|   Jane Smith| 25|Female| 45000|15-02-2016|
|          3|          102|    Bob Brown| 35|  Male| 55000|01-05-2014|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|
|          5|          103|    Jack Chan| 40|  Male| 60000|01-04-2013|
|          6|          103|    Jill Wong| 32|Female| 52000|01-07-2018|
|          7|          101|James Johnson| 42|  Male| 70000|15-03-2012|
|          8|          102|     Kate Kim| 29|Female| 51000|01-10-2019|
|          9|          103|      Tom Tan| 33|  Male| 58000|01-06-2016|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|
|         11|          104|   David Park| 38|  Male| 65000|01-11-2015|
|     

In [5]:
dept_df.show()

+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|department_id|     department_name|         description|                city|state|            country|
+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|            1|         Bryan-James|Optimized disinte...|        Melissaburgh|   FM|Trinidad and Tobago|
|            2|Smith, Craig and ...|Digitized empower...|          Morrisside|   DE|          Sri Lanka|
|            3|Pittman, Hess and...|Multi-channeled c...|         North David|   SC|       Turkmenistan|
|            4|Smith, Snyder and...|Reactive neutral ...|       Lake Jennifer|   TX|         Madagascar|
|            5|          Hardin Inc|Re-contextualized...|           Hayestown|   WA|               Fiji|
|            6|         Sanders LLC|Innovative multim...|         Phamchester|   TN|         Micronesia|
|            7|         Ward-Gordon|Progressive logis..

In [13]:
emp_df.rdd.getNumPartitions()

1

In [15]:
emp_partitioned=emp_df.repartition(4)
emp_partitioned.rdd.getNumPartitions()

4

In [19]:
from pyspark.sql.functions import *
emp_partitioned.withColumn('partition_id', spark_partition_id()).show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|         11|          104|   David Park| 38|  Male| 65000|01-11-2015|           0|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|           0|
|         19|          103|  Steven Chen| 36|  Male| 62000|01-08-2015|           0|
|          9|          103|      Tom Tan| 33|  Male| 58000|01-06-2016|           0|
|         13|          106|    Brian Kim| 45|  Male| 75000|01-07-2011|           0|
|          1|          101|     John Doe| 30|  Male| 50000|01-01-2015|           1|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|           1|
|         17|          105|  George Wang| 34|  Male| 57000|15-03-2016|           1|
|         20|          102|    Grace Kim| 32|Female| 53000|01-11-2018|      

In [17]:
emp_repartitioned= emp_df.repartition(4, "department_id").withColumn('partition_id', spark_partition_id())
emp_repartitioned.show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|          3|          102|    Bob Brown| 35|  Male| 55000|01-05-2014|           0|
|          4|          102|    Alice Lee| 28|Female| 48000|30-09-2017|           0|
|          8|          102|     Kate Kim| 29|Female| 51000|01-10-2019|           0|
|         14|          107|    Emily Lee| 26|Female| 46000|01-01-2019|           0|
|         16|          107|  Kelly Zhang| 30|Female| 49000|01-04-2018|           0|
|         20|          102|    Grace Kim| 32|Female| 53000|01-11-2018|           0|
|         12|          105|   Susan Chen| 31|Female| 54000|15-02-2017|           1|
|         17|          105|  George Wang| 34|  Male| 57000|15-03-2016|           1|
|         10|          104|     Lisa Lee| 27|Female| 47000|01-08-2018|      

In [21]:
emp_repartitioned.groupBy("partition_id").count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    6|
|           1|    2|
|           2|    5|
|           3|    7|
+------------+-----+

